# LangChain: Memory

## Goal of this notebook

Large Language Models do not have persistent memory by default.  
Each request to the model is stateless unless we explicitly pass previous messages.

The goal of this notebook is to explore how **memory mechanisms** can be implemented in LangChain in order to build conversational applications.

We will explore several memory strategies:

1. Storing full conversation history
2. Limiting memory using a sliding window
3. Limiting memory by token size
4. Compressing conversation history with summarization

These techniques are fundamental for building chatbots, assistants and agent systems.


In [1]:
# Install dependencies (run if needed)
# Updated for 2026 LangChain modular packages
# %pip install -q -U langchain langchain-core langchain-community langchain-openai faiss-cpu python-dotenv


Note: you may need to restart the kernel to use updated packages.


## Environment Setup

In this section we install the required dependencies and configure the environment.

We will use:
- LangChain
- OpenAI chat models
- LangChain message history utilities

This setup allows us to build conversational pipelines with memory.


In [3]:
import os
import warnings
from dotenv import load_dotenv

load_dotenv()

warnings.filterwarnings("ignore")

from openai import OpenAI

client = OpenAI()

In [2]:
llm_model = "gpt-4o-mini"

## Building a Conversational Chain with Memory

Here we create a simple conversational pipeline using LangChain.

The key idea is that we attach a **message history object** to the chain so that the model can access previous conversation turns.

This allows the model to answer context-dependent questions.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


In [5]:
llm = ChatOpenAI(
    model=llm_model,
    temperature=0,
)

# Build a conversational LCEL pipeline and attach stateful chat history
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful, conversational assistant."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)
parser = StrOutputParser()
base_chain = prompt | llm | parser

# Use an in-memory chat history store keyed by session id
_store = {}

def get_session_history(session_id: str):
    """Return (and lazily create) the chat history for a given session."""
    if session_id not in _store:
        _store[session_id] = InMemoryChatMessageHistory()
    return _store[session_id]

# RunnableWithMessageHistory automatically reads/writes messages to the history
# It will pull `session_id` from config["configurable"]["session_id"]
chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    output_messages_key="output",
)

# Default "session" id for this notebook section
session_id = "andrew-session"
# Expose the underlying history object so we can inspect it later in the notebook
memory = get_session_history(session_id)

In [6]:
chain_with_history.invoke(
    {"input": "Hi, my name is Andrew"},
    config={"configurable": {"session_id": session_id}},
)

'Hi Andrew! How can I assist you today?'

In [7]:
chain_with_history.invoke(
    {"input": "What is 1+1?"},
    config={"configurable": {"session_id": session_id}},
)

'1 + 1 equals 2. If you have any more questions or need help with something else, feel free to ask!'

In [8]:
chain_with_history.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": session_id}},
)

'Your name is Andrew. How can I help you today, Andrew?'

## Inspecting the Stored Messages

The conversation history is stored as a list of messages.

Each message contains:
- the role (user / assistant)
- the message content

By inspecting this history we can verify that the memory buffer correctly stores the conversation.

In [9]:
# `memory` is now an InMemoryChatMessageHistory. Inspect the buffered messages:
for m in memory.messages:
    print(f"{m.type}: {m.content}")

human: Hi, my name is Andrew
ai: Hi Andrew! How can I assist you today?
human: What is 1+1?
ai: 1 + 1 equals 2. If you have any more questions or need help with something else, feel free to ask!
human: What is my name?
ai: Your name is Andrew. How can I help you today, Andrew?


In [10]:
# Modern equivalent of `load_memory_variables`: wrap messages in a dict
{"history": memory.messages}

{'history': [HumanMessage(content='Hi, my name is Andrew', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hi Andrew! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is 1+1?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='1 + 1 equals 2. If you have any more questions or need help with something else, feel free to ask!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Your name is Andrew. How can I help you today, Andrew?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

## Buffer Memory

The simplest memory strategy is to store **the entire conversation history**.

This is known as **buffer memory**.

Advantages:
- Simple to implement
- Preserves full context

Limitations:
- Memory grows indefinitely
- Token limits can be exceeded

In [11]:
# Legacy `ConversationBufferMemory` example rewritten using `InMemoryChatMessageHistory`
buffer_memory = InMemoryChatMessageHistory()

In [12]:
buffer_memory.add_user_message("Hi")
buffer_memory.add_ai_message("What's up")

In [13]:
buffer_text = "\n".join(f"{m.type}: {m.content}" for m in buffer_memory.messages)
print(buffer_text)

human: Hi
ai: What's up


In [14]:
{"history": buffer_memory.messages}

{'history': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
  AIMessage(content="What's up", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [15]:
buffer_memory.add_user_message("Not much, just hanging")
buffer_memory.add_ai_message("Cool")

In [16]:
{"history": buffer_memory.messages}

{'history': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
  AIMessage(content="What's up", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Cool', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

## Window Memory (Sliding Window)

Instead of storing the entire conversation, we can keep only the **last K messages**.

This approach is useful when:

- conversations become very long
- token limits are a concern

The model receives only the most recent interaction context.

In [17]:
# Updated imports to modern chat history + runnable history APIs
from langchain_core.messages import trim_messages
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

history = InMemoryChatMessageHistory()

history.add_user_message("Hi")
history.add_ai_message("Hello!")

messages = trim_messages(
    history.messages,
    max_tokens=200,
    strategy="last",
    token_counter="approximate",
)

In [18]:
# Simple windowed memory using `InMemoryChatMessageHistory` and manual slicing
from langchain_core.chat_history import InMemoryChatMessageHistory

window_history = InMemoryChatMessageHistory()
# Number of previous exchanges (user+AI pairs) to keep in the "window"
k_window = 1

In [19]:
window_history.add_user_message("Hi")
window_history.add_ai_message("What's up")
window_history.add_user_message("Not much, just hanging")
window_history.add_ai_message("Cool")


In [20]:
# Mimic `ConversationBufferWindowMemory(k=1)` by taking the last 2 messages
{"history": window_history.messages[-2 * k_window:]}

{'history': [HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Cool', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [21]:
llm_window = ChatOpenAI(temperature=0.0, model=llm_model)

# LCEL pipeline for this section (no legacy `ConversationBufferWindowMemory`)
prompt_window = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful, conversational assistant."),
        ("human", "{input}"),
    ]
)
parser_window = StrOutputParser()
window_chain = prompt_window | llm_window | parser_window

In [22]:
window_chain.invoke("Hi, my name is Andrew")

'Hi Andrew! How can I assist you today?'

In [23]:
window_chain.invoke("What is 1+1?")

'1 + 1 equals 2.'

In [24]:
window_chain.invoke("What is my name?")

"I don't know your name. If you'd like to share it, feel free!"

## Token-Limited Memory

Another strategy is to limit the memory by the **number of tokens** rather than the number of messages.

LangChain provides utilities such as `trim_messages` that allow us to keep only the part of the conversation that fits within a token budget.

This helps control API costs and prevents context overflow.

In [25]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [26]:
# Token-limited memory using `trim_messages` instead of `ConversationTokenBufferMemory`
# (requires `trim_messages` and `InMemoryChatMessageHistory` imports from earlier cells)

token_history = InMemoryChatMessageHistory()

token_history.add_user_message("AI is what?!")
token_history.add_ai_message("Amazing!")

token_history.add_user_message("Backpropagation is what?")
token_history.add_ai_message("Beautiful!")

token_history.add_user_message("Chatbots are what?")
token_history.add_ai_message("Charming!")

In [27]:
trimmed_messages = trim_messages(
    token_history.messages,
    max_tokens=100,
    strategy="last",
    token_counter="approximate",
)
{"history": trimmed_messages}

{'history': [HumanMessage(content='AI is what?!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Amazing!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Backpropagation is what?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Beautiful!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Chatbots are what?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Charming!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

## Summary Memory

When conversations become very long, storing all messages becomes inefficient.

An alternative approach is to **summarize the conversation history** and keep only a compressed version.

This allows the model to retain the important context while significantly reducing the number of tokens.

In [28]:
# Modern summary "memory" example using LCEL instead of `ConversationSummaryBufferMemory`
from langchain_core.chat_history import InMemoryChatMessageHistory

In [29]:
# create a long string
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

# Store the conversation turns in chat history instead of using ConversationSummaryBufferMemory
summary_history = InMemoryChatMessageHistory()
summary_history.add_user_message("Hello")
summary_history.add_ai_message("What's up")
summary_history.add_user_message("Not much, just hanging")
summary_history.add_ai_message("Cool")
summary_history.add_user_message("What is on the schedule today?")
summary_history.add_ai_message(schedule)

In [30]:
# Show the stored conversation turns
{"history": summary_history.messages}

{'history': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}),
  AIMessage(content="What's up", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Cool', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is on the schedule today?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.', additional_kwargs={}, response_metadata={}, tool_

In [31]:
# Build an LCEL pipeline that summarizes the conversation history
summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that summarizes conversations."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "Please provide a short summary of our conversation."),
    ]
)
summary_parser = StrOutputParser()
summary_chain = summary_prompt | llm | summary_parser

In [32]:
summary = summary_chain.invoke({"history": summary_history.messages})
summary

'You greeted me and mentioned you were just hanging out. I then provided you with a schedule for the day, which includes a meeting with the product team, time to work on your LangChain project, and a lunch meeting with a customer to discuss the latest in AI. I also reminded you to bring your laptop for a demo.'

In [33]:
# The original (unsummarized) history is still available
{"history": summary_history.messages}

{'history': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}),
  AIMessage(content="What's up", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Cool', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is on the schedule today?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.', additional_kwargs={}, response_metadata={}, tool_

## Conclusion

In this notebook we explored several approaches to managing memory in LangChain-based conversational systems.

Key takeaways:

- LLMs are stateless by default, so conversation history must be explicitly managed.
- Different memory strategies can be used depending on the application:
  - buffer memory for full context
  - window memory for recent interactions
  - token-limited memory to control context size
  - summary memory to compress long conversations.
  
Understanding these techniques is essential for building scalable chatbots, assistants and agent-based systems.